# **Load Files to Bronze Tables**

### This notebook takes a table name as a parameter:
1. To construct a file path variable to determine where the JSON data is stored
2. For naming the destination/sink table in the Lakehouse

### Special notes:
- option("multiline", "true") was needed in order to read some of the JSON files
- spark.conf.set("spark.sql.caseSensitive", True) command is used to allow upper case characters in table names
- option('delta.columnMapping.mode' , 'name') option in the write enables spaces in column names



## Parameters

In [1]:
# Parameter value(s) supplied by calling pipeline or notebook

# Default values 

# Name of schema/directory location
SCHEMA_NAME = "blog"
# Name of file/table being processed
TABLE_NAME = "Fabric_Updates"

# Get the schema of the input dataframe as a JSON structure. Set to True for first run or when schema changes
MAP_STRUCTURE = False

# Boolean indicating if a schema is going to be applied
# Typically we will want to apply a schema, but it may be handy to turn this off to debug data mapping errors
PROCESS_SCHEMA_FILE = True

# File type/extension of the incoming data source file
FILE_TYPE = 'json'

# Format used by the source for Date/Time fields
#timestamp_format = "LL/dd/yyyy HH:mm:ss"
TIMESTAMP_FORMAT = "yyyy-LL-dd HH:mm:ss"

#input_file_path = "Files/" + SCHEMA_NAME + "/Data/"  + TABLE_NAME + "." + FILE_TYPE

# Boolean field.
# Setting "debug" to True helps prevent accidentally overwriting data when debugging
# calling pipeline, notebook, or DAG should set "debug" to False in order to write the final dataframe 
DEBUG = False

StatementMeta(, 2ac5a5b2-0b0f-4bd6-9b23-2872d0f91288, 3, Finished, Available, Finished)

## Notebook level Libraries, Settings, and Variables

In [ ]:
### Libraries ###
from pyspark.sql.types import *
from pyspark.sql.functions import col, lit
import json

# Setting Spark SQL casesensitive to "True" allows for case sensitive table names
spark.conf.set("spark.sql.caseSensitive", True)

### Set variables ###

# Build string with the file path for reading the data
input_file_path = "Files/" + SCHEMA_NAME + "/Data/"  + TABLE_NAME + "." + FILE_TYPE

# Build string with the file path for reading the schema
schema_file = "Files/" + SCHEMA_NAME + "/Schemas/" + TABLE_NAME.replace("_Full","").replace("_Incremental","") + "_Schema.json"

# Build string with for naming sink table
save_as_name = "Sandbox_LH_Bronze." + SCHEMA_NAME.lower() + "." + TABLE_NAME.replace("_","").replace("Full","").replace("Incremental","") # Remove underscore characters in table name


print(input_file_path)
print(schema_file)
print(save_as_name)

## Initial Statistics

In [5]:
# How many rows in the original table.

StatementMeta(, 2ac5a5b2-0b0f-4bd6-9b23-2872d0f91288, 7, Finished, Available, Finished)

## Read Data

In [ ]:
# Load the file into the dataframe 
if FILE_TYPE.lower() == "json":
    # Load JSON data to dataframe
    df = spark.read.option("multiline", "true").option('timestampFormat', TIMESTAMP_FORMAT).json(input_file_path)
elif FILE_TYPE.lower() == "csv":
    # Load CSV data to dataframe
    df = spark.read.format("csv").option('timestampFormat', "yyyy-MM-dd HH:mm:ss").option('header','true').load(csv_file_path)
else:
    raise ValueError(f"Unsupported FILE_TYPE: {FILE_TYPE}")

display(df)

## Process Schema file

If "map_structure" parameter is set to "True", then read and process the schema file for use when saving the data.

In [ ]:
# Optional - Generate Schemas to save as JSON
# When you bring in a new file for the first time, uncomment the step below to generate the STRUCT in JSON format which can be used with VS Code.
if MAP_STRUCTURE:
    df.schema.json()

In [ ]:
display(df)

In [ ]:
#casted_df.printSchema

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, lit, to_timestamp, to_date
from pyspark.sql.types import (
    StructType,
    TimestampType,
    DateType,
    DecimalType,
    DoubleType,
)

#global_timestamp_format: str = "MM/dd/yyyy HH:mm:ss",


def cast_dataframe_to_schema(
    df: DataFrame,
    schema_struct: StructType,
    #global_timestamp_format: str = "yyyy/MM/dd H:mm",
    global_timestamp_format: str = "yyyy-MM-dd HH:mm:ss",
    field_format_overrides: dict = None
) -> DataFrame:
    """
    Casts columns in a DataFrame to the types defined in the given schema.

    Parameters:
        df (DataFrame): The input Spark DataFrame with raw string data.
        schema_struct (StructType): Target schema definition with Spark data types.
        global_timestamp_format (str): Default datetime format for casting.
        field_format_overrides (dict): Optional per-column format overrides.

    Returns:
        DataFrame: A new DataFrame with columns cast to the desired schema types.
    """
    if field_format_overrides is None:
        field_format_overrides = {}

    casted_columns = []
    for field in schema_struct.fields:
        col_name = field.name
        target_type = field.dataType

        if col_name in df.columns:
            # Column exists
            if isinstance(target_type, TimestampType):
                fmt = field_format_overrides.get(col_name, global_timestamp_format)
                casted_columns.append(to_timestamp(col(col_name), fmt).alias(col_name))

            elif isinstance(target_type, DateType):
                fmt = field_format_overrides.get(col_name, global_timestamp_format)
                casted_columns.append(to_date(col(col_name), fmt).alias(col_name))

            elif isinstance(target_type, DecimalType):
                casted_columns.append(col(col_name).cast(target_type).alias(col_name))

            elif isinstance(target_type, DoubleType):
                casted_columns.append(col(col_name).cast(DoubleType()).alias(col_name))

            else:
                casted_columns.append(col(col_name).cast(target_type).alias(col_name))
        else:
            # Column missing — add NULL of the correct type
            if isinstance(target_type, TimestampType):
                fmt = field_format_overrides.get(col_name, global_timestamp_format)
                casted_columns.append(to_timestamp(lit(None), fmt).alias(col_name))

            elif isinstance(target_type, DateType):
                fmt = field_format_overrides.get(col_name, global_timestamp_format)
                casted_columns.append(to_date(lit(None), fmt).alias(col_name))

            elif isinstance(target_type, DecimalType):
                casted_columns.append(lit(None).cast(target_type).alias(col_name))

            elif isinstance(target_type, DoubleType):
                casted_columns.append(lit(None).cast(DoubleType()).alias(col_name))

            else:
                casted_columns.append(lit(None).cast(target_type).alias(col_name))

    return df.select(casted_columns)


 #global_timestamp_format="MM/dd/yyyy HH:mm:ss",    

In [ ]:
schema_file = "/lakehouse/default/Files/blog/Schemas/Fabric_Updates_Schema.json"

if PROCESS_SCHEMA_FILE:

    # Read Schema
    with open(f"{schema_file}","r") as file:
        schema = json.load(file)

    # Change Schema definition to Struct
    schema_struct = StructType.fromJson(schema)


    casted_df = cast_dataframe_to_schema(
    df,
    schema_struct,
    #global_timestamp_format:="yyyy/MM/dd H:mm",
    global_timestamp_format := "yyyy-MM-dd HH:mm:ss",
   
    field_format_overrides={
        #  "DateOfBirth": "yyyy-MM-dd H:mm:ss",
         "date": "yyyy-MM-dd",
         "fetched_at": "yyyy-MM-dd'T'HH:mm:ss'Z'"
     }
)


display(casted_df)

## Standardize Column Names

In [ ]:
import re
from pyspark.sql import DataFrame

# Define acronyms to preserve (case-insensitive matching)
ACRONYMS = ["DOB", "SSN"]

def isolate_acronyms(name: str) -> str:
    # Surround known acronyms with spaces if not already separated
    for acronym in ACRONYMS:
        pattern = re.compile(rf'(?i)(?<!\w)({acronym})(?!\w)')
        name = pattern.sub(f' {acronym} ', name)

        # Handle embedded acronyms like "customerDOBDate" → "customer DOB Date"
        name = re.sub(rf'(?i)(?<!\s)({acronym})(?!\s)', r' \1 ', name)
    return name

def split_and_format_column_name(name: str, title_case: bool = False) -> str:
    # Step 0: Normalize and preserve acronyms
    name = isolate_acronyms(name)
    
    # Step 1: Replace underscores with spaces
    name = name.replace('_', ' ')

    # Step 1a: Remove forward slashes (let capitalization logic insert spaces later)
    name = name.replace('/', '')

    # Step 2: Insert space before capital letter after lowercase or digit
    name = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', ' ', name)

    # Step 3: Insert space between acronym and word (e.g., "SSNDate" → "SSN Date")
    name = re.sub(r'(?<=[A-Z])(?=[A-Z][a-z])', ' ', name)

    # Step 4: Insert space between letter and digit (e.g., "Line1" → "Line 1")
    name = re.sub(r'(?<=[A-Za-z])(?=\d)', ' ', name)

    # Step 5: Normalize spacing
    name = re.sub(r'\s+', ' ', name).strip()

    # Step 6: Title-case if needed, but preserve acronyms in full caps
    if title_case:
        words = name.split()
        name = ' '.join([word if word.upper() in ACRONYMS else word.title() for word in words])

    return name

def rename_columns_with_spaces(df: DataFrame, title_case: bool = True) -> DataFrame:
    new_column_names = [split_and_format_column_name(col, title_case) for col in df.columns]
    return df.toDF(*new_column_names)

# Call the custom rename function
df = rename_columns_with_spaces(casted_df)

display(df)


In [13]:
#casted_df.printSchema

StatementMeta(, 2ac5a5b2-0b0f-4bd6-9b23-2872d0f91288, 15, Finished, Available, Finished)

In [14]:
if not DEBUG:
    # Write dataframe to Lakehouse table
    df.write.format("delta").mode("overwrite").option('delta.columnMapping.mode' , 'name').saveAsTable(save_as_name) 

StatementMeta(, 2ac5a5b2-0b0f-4bd6-9b23-2872d0f91288, 16, Finished, Available, Finished)

## Final Statistics

## Resources

##### Using Struct to apply schema to CSV files on read.
https://microsoftlearning.github.io/mslearn-fabric/Instructions/Labs/03b-medallion-lakehouse.html




